# Mission 01: KPI Dashboard Makeover

The operations team of a taxi company uses the dashboard below every Monday. Nobody trusts it: the numbers are hard to read, the colours are confusing and one chart is misleading. Your job is to **redesign** it for the February 2024 review.

**Lessons used:** 01 (principles), 03 (colour and accessibility), 07 (dashboards), 15 (KPI definitions), 17 (QA).

**Data:** NYC yellow taxi trips, February 2024 (cleaned).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets

COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "trip_distance",
        "fare_amount", "tip_amount", "total_amount", "payment_type"]

def load_month(month: str) -> pd.DataFrame:
    df = pd.read_parquet(datasets.taxi(month), columns=COLS)
    start = pd.Timestamp(month + "-01")
    dur = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    ok = (df["tpep_pickup_datetime"].between(start, start + pd.offsets.MonthBegin(1), inclusive="left")
          & dur.between(1, 180) & df["trip_distance"].between(0.1, 100) & df["fare_amount"].between(3, 500))
    return df[ok].assign(duration_min=dur[ok])

zones = pd.read_csv(datasets.fetch("taxi_zone_lookup.csv"))
trips = load_month("2024-02")
trips["borough"] = trips["PULocationID"].map(zones.set_index("LocationID")["Borough"])

def bad_dashboard(trips):
    """The current dashboard of the operations team (do not change this function)."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    b = trips.groupby("borough").size()
    axes[0, 0].pie(b, labels=b.index, autopct="%1.0f%%", colors=plt.cm.rainbow(np.linspace(0, 1, len(b))))
    d = trips.groupby(trips["tpep_pickup_datetime"].dt.day)["total_amount"].sum()
    axes[0, 1].bar(d.index, d.values, color="#FFE45C")
    axes[0, 1].set_ylim(d.min() * 0.95, d.max())
    h = trips.groupby([trips["tpep_pickup_datetime"].dt.hour, "PULocationID"]).size().unstack().iloc[:, :15]
    h.plot(ax=axes[1, 0], legend=True)
    axes[1, 0].legend(fontsize=5, ncol=3)
    p = trips.groupby("payment_type")["tip_amount"].mean()
    ax2 = axes[1, 1].twinx()
    axes[1, 1].bar(p.index, p.values, color="red")
    ax2.plot(p.index, trips.groupby("payment_type").size(), color="green")
    fig.suptitle("DASHBOARD")
    return fig

fig = bad_dashboard(trips)
plt.show()

## Tasks

1. **Critique.** List at least six problems of the current dashboard, each with the principle it breaks.
2. **Define KPIs.** Choose 4–5 KPIs for the weekly operations review. For each, write the definition, the comparison (previous period) and the "good" direction.
3. **Redesign.** Build a new dashboard figure: KPI tiles, a main trend, and two breakdowns. Use message titles, a colour-blind-safe palette and direct labels.
4. **QA.** Write a linter check (or reuse the one from Lesson 17) and run it on the old and the new dashboard. Write two reconciliation tests for your charts' numbers.
5. **Explain.** In five sentences, explain to the operations manager what changed and why.

In [ ]:
# Tasks 1-5
...

## Submission Checklist

- [ ] Every problem is linked to a principle.
- [ ] KPIs have definitions, comparisons and directions.
- [ ] The new dashboard passes your linter and your reconciliation tests.
- [ ] Titles state findings; the default view answers the main question.

A solution is in `01-KPI-Dashboard-Makeover-solution.ipynb`. Try the mission first.